# Phase 4: SFT — Supervised Fine-tuning for Tamil Instruction Following

**Goal**: Train the CPT model to follow Tamil instructions using synthetic + curated data.

**Starts from**: `wickkiey/tamil-llama-3.1-8b-cpt-v1` (Phase 1 checkpoint)  
**Hardware**: Colab A100 40GB  
**Output**: `wickkiey/tamil-llama-3.1-8b-sft-v1`

**Training data** (all free):
- `wickkiey/tamil-synthetic-instructions` — 50K synthetic pairs (Phase 3)
- `ai4bharat/IndicInstruct` — Tamil subset
- `ai4bharat/IndicXNLI` — Tamil NLI
- `ai4bharat/IndicSentiment` — Tamil sentiment

In [ ]:
import subprocess, sys
for pkg in ["unsloth", "trl", "datasets", "peft", "accelerate", "bitsandbytes"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Ready.")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
CPT_CHECKPOINT  = "wickkiey/tamil-llama-3.1-8b-cpt-v1"   # Phase 1 output
MAX_SEQ_LEN     = 2048
NUM_EPOCHS      = 3
BATCH_SIZE      = 2
GRAD_ACCUM      = 8
LORA_RANK       = 128
LORA_ALPHA      = 256

OUTPUT_DIR      = "outputs/sft_v1"
HF_REPO         = "wickkiey/tamil-llama-3.1-8b-sft-v1"
HF_TOKEN        = None

# Tamil Alpaca system prompt
SYSTEM_PROMPT = (
    "நீங்கள் ஒரு பயனுள்ள தமிழ் மொழி உதவியாளர். "
    "கேட்கப்படும் கேள்விகளுக்கு தமிழில் தெளிவான பதில் தருங்கள்."
)

print(f"CPT checkpoint : {CPT_CHECKPOINT}")
print(f"LoRA rank      : {LORA_RANK}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# ── Load model from CPT checkpoint ────────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CPT_CHECKPOINT,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = 0,
    bias                = "none",
    use_gradient_checkpointing = "unsloth",
    random_state        = 42,
)

model.print_trainable_parameters()

In [ ]:
# ── Load and format training datasets ─────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets, Dataset

def format_alpaca(instruction, output, input_text=""):
    """Format as Alpaca-style Tamil prompt."""
    if input_text:
        return (
            f"### கட்டளை:\n{instruction}\n\n"
            f"### உள்ளீடு:\n{input_text}\n\n"
            f"### பதில்:\n{output}"
        )
    return (
        f"### கட்டளை:\n{instruction}\n\n"
        f"### பதில்:\n{output}"
    )

EOS = tokenizer.eos_token
all_texts = []

# 1. Synthetic instructions (Phase 3)
try:
    synthetic_ds = load_dataset("wickkiey/tamil-synthetic-instructions", split="train")
    for row in synthetic_ds:
        text = format_alpaca(row["instruction"], row["output"], row.get("input", ""))
        all_texts.append({"text": text + EOS})
    print(f"Synthetic instructions: {len(synthetic_ds):,}")
except Exception as e:
    print(f"Synthetic dataset not yet available: {e}")
    print("Run evaluation/01_generate_synthetic_local.ipynb first.")

# 2. IndicInstruct Tamil subset
try:
    indic_inst = load_dataset("ai4bharat/IndicInstruct", split="train")
    ta_subset  = indic_inst.filter(lambda x: x.get("lang", "") == "ta" or x.get("language", "") == "ta")
    for row in ta_subset:
        inst   = row.get("instruction", row.get("input", ""))
        output = row.get("output", row.get("response", ""))
        if inst and output:
            all_texts.append({"text": format_alpaca(inst, output) + EOS})
    print(f"IndicInstruct Tamil: {len(ta_subset):,}")
except Exception as e:
    print(f"IndicInstruct: {e}")

# 3. IndicXNLI Tamil — NLI as instruction ("Does premise entail hypothesis?")
try:
    xnli = load_dataset("ai4bharat/IndicXNLI", "ta", split="train")
    label_map = {0: "உட்குறிப்பு (entailment)", 1: "நடுநிலை (neutral)", 2: "முரண்பாடு (contradiction)"}
    for row in xnli:
        inst = (
            f"கீழ்க்கண்ட இரண்டு வாக்கியங்களுக்கு இடையேயான தொடர்பை கண்டுபிடியுங்கள்.\n"
            f"முன்வைப்பு: {row['premise']}\n"
            f"அனுமானம்: {row['hypothesis']}"
        )
        out = label_map.get(row["label"], "தெரியவில்லை")
        all_texts.append({"text": format_alpaca(inst, out) + EOS})
    print(f"IndicXNLI Tamil: {len(xnli):,}")
except Exception as e:
    print(f"IndicXNLI: {e}")

# 4. IndicSentiment Tamil — classify sentiment
try:
    sent = load_dataset("ai4bharat/IndicSentiment", "ta", split="train")
    label_map_s = {"Positive": "நேர்மறை", "Negative": "எதிர்மறை", "Neutral": "நடுநிலை"}
    for row in sent:
        text_s = row.get("text", row.get("sentence", ""))
        lbl    = label_map_s.get(row.get("label", ""), row.get("label", ""))
        if text_s and lbl:
            inst = f"கீழ்க்கண்ட வாக்கியத்தின் உணர்வை கண்டுபிடியுங்கள்:\n{text_s}"
            all_texts.append({"text": format_alpaca(inst, lbl) + EOS})
    print(f"IndicSentiment Tamil: {len(sent):,}")
except Exception as e:
    print(f"IndicSentiment: {e}")

dataset = Dataset.from_list(all_texts).shuffle(seed=42)
print(f"\nTotal SFT training samples: {len(dataset):,}")

In [ ]:
# ── Trainer ────────────────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LEN,
    dataset_num_proc   = 2,
    packing            = False,   # False for instruction tuning (preserve turn boundaries)
    args = TrainingArguments(
        output_dir                  = OUTPUT_DIR,
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio                = 0.03,
        learning_rate               = 2e-5,          # lower LR for SFT than CPT
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 10,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = 42,
        save_strategy               = "steps",
        save_steps                  = 200,
        save_total_limit            = 2,
        report_to                   = "none",
    ),
)

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────────
trainer_stats = trainer.train()
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f}s")

In [ ]:
# ── Inference test ─────────────────────────────────────────────────────────────
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

test_prompts = [
    "### கட்டளை:\nதமிழ்நாட்டின் தலைநகரம் எது?\n\n### பதில்:\n",
    "### கட்டளை:\nசூரிய மண்டலத்தில் உள்ள கோள்களின் பெயர்களை கூறுங்கள்.\n\n### பதில்:\n",
    "### கட்டளை:\nநீர் சேமிப்பு முக்கியம் என்பதை ஒரு பத்தியில் விளக்குங்கள்.\n\n### பதில்:\n",
]

for prompt in test_prompts:
    inputs  = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs, max_new_tokens=150, temperature=0.7,
        do_sample=True, pad_token_id=tokenizer.eos_token_id,
    )
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("-" * 60)

In [ ]:
# ── Save & push ────────────────────────────────────────────────────────────────
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

try:
    from google.colab import drive
    model.save_pretrained(f"/content/drive/MyDrive/Tamil-LLM/{OUTPUT_DIR}")
    tokenizer.save_pretrained(f"/content/drive/MyDrive/Tamil-LLM/{OUTPUT_DIR}")
    print("Saved to Google Drive.")
except ImportError:
    pass

model.push_to_hub(HF_REPO, token=HF_TOKEN,
    commit_message="SFT v1 — Tamil instruction tuning on CPT checkpoint")
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"Pushed: https://huggingface.co/{HF_REPO}")

## Next Step
Proceed to `finetuning/03_dpo_preference_training.ipynb` to improve response quality via preference learning.